# Baseline Classifier

Out-of-sample, walk-forward logistic regression predicting the **quantile / binary `label`**
built on the Feature Engineering page. All logic lives in `irp.models.classifier`.

**The dataset comes from `/features`**: build features + a **Quantile** (or Up/Down) label,
then **Export parquet**. This notebook loads that export — it does not rebuild the data.

In [1]:
from irp.models import classifier as clf
from sklearn.linear_model import LogisticRegression

# ── parameters ──
EXPORT_PATH = 'features_20260601_034025.parquet'        # None = most recent; or: clf.list_exports().iloc[1]['file']
FEATURE_COLS = None       # None = infer (numeric cols except Date/Ticker/fwd_ret/label)
MODEL = LogisticRegression(max_iter=10_000)
MIN_TRAIN_DATES = 12
TPL = clf.nb_template()

clf.list_exports()        # row 0 = newest — set EXPORT_PATH to pick a different one

,file,modified,mb
0,classifier_demo_20260601_110810.parquet,2026-06-01 11:08:10.482964,5.10
1,features_20260601_034025.parquet,2026-06-01 03:40:29.998514,383.37
2,features_20260601_003532.parquet,2026-06-01 00:35:37.334741,430.52


## 1 · Load dataset (must carry a quantile/binary `label`)

In [2]:
df, features = clf.load_export(EXPORT_PATH, feature_cols=FEATURE_COLS, target='label')
print(len(features), 'features:', features)
df['label'].value_counts().sort_index()

loaded features_20260601_034025.parquet  3,985,529 rows × 48 cols
44 features: ['rsi_14', 'macd_hist', 'macd_norm', 'bb_pct', 'ma7_ma28', 'ma14_ma56', 'gross_margin', 'op_margin', 'net_margin', 'roe', 'roa', 'roic', 'fcf_margin', 'asset_turnover', 'cfo_ni_ratio', 'accruals', 'revenue', 'net_income', 'total_assets', 'total_equity', 'op_cashflow', 'rand', 'rev_growth_1y', 'earn_growth_1y', 'debt_equity', 'net_debt_ebitda', 'interest_coverage', 'piotroski_fscore', 'close', 'close_lag1', 'close_lag2', 'close_lag3', 'close_lag4', 'close_lag5', 'close_lag6', 'close_lag7', 'volume', 'volume_lag1', 'volume_lag2', 'volume_lag3', 'volume_lag4', 'volume_lag5', 'volume_lag6', 'volume_lag7']


label
0.0    1291517
1.0    1290613
2.0    1290828
Name: count, dtype: int64

## 2 · Walk-forward classification
Expanding window: fit on the past, predict the current cross-section's class (no look-ahead).

In [ ]:
res = clf.walk_forward_classifier(df, features, target='label', model=MODEL, min_train_dates=MIN_TRAIN_DATES)
_ = clf.summary(res)

/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/mnt/Dev/active_python_projects/investment_research_platform/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also

## 3 · Visualize
Confusion + accuracy. Quintiles by predicted score need `fwd_ret` in the export (Quantile mode keeps it).

In [ ]:
clf.plot_confusion(res, TPL)

In [ ]:
clf.plot_accuracy(res, TPL)

In [ ]:
clf.plot_quintiles(res, TPL)